# 03 - Debugging Scenarios

## Purpose

Document realistic debugging scenarios from the capstone project.

Each scenario explains the symptom, likely cause, investigation steps, fix, prevention, and final result.

## Required scenarios

At least three debugging scenarios should be documented.

## Main idea

Debugging should be structured. The goal is to classify the issue before changing code randomly.

In [0]:
from pyspark.sql import Row

In [0]:
debugging_scenarios = [
    Row(
        scenario_id="DBG-001",
        category="Path",
        symptom="Auto Loader or file copy fails with path not found.",
        likely_cause="The notebook points to the wrong workspace path, tutorial repo path, or Unity Catalog volume path.",
        investigation="Print the source path and target path, then use dbutils.fs.ls() to confirm that the folder exists.",
        fix="Correct the path and make sure files are copied from the tutorial repo into the capstone repo and then into the landing volume.",
        prevention="Always print landing paths, checkpoint paths, schema paths, and source paths before ingestion.",
        project_example="During Bronze setup, source sample_data paths had to be corrected before copying files into landing volumes.",
    ),
    Row(
        scenario_id="DBG-002",
        category="Schema",
        symptom="Silver grid transformation fails because a column does not exist.",
        likely_cause="The tutorial example expected a column name that differs from the actual Bronze schema.",
        investigation="Print bronze_grid_df.columns and inspect sample rows before applying the transform.",
        fix="Update the reusable transform module to use the actual Bronze column event_date instead of event_ts.",
        prevention="Inspect real source schemas before copying transformation logic from a tutorial.",
        project_example="The grid event transform was updated to derive event_timestamp from event_date.",
    ),
    Row(
        scenario_id="DBG-003",
        category="Python module cache",
        symptom="A notebook still uses an old version of a Python transform after the file was updated.",
        likely_cause="Databricks keeps the imported module in memory during the notebook session.",
        investigation="Check that the .py file is saved correctly and reload the module with importlib.reload().",
        fix="Use importlib.reload(module) and call functions through the reloaded module object.",
        prevention="After editing files under src/, reload modules or restart the compute before rerunning dependent cells.",
        project_example="The grid event notebook needed importlib.reload() after updating grid_event_transforms.py.",
    ),
    Row(
        scenario_id="DBG-004",
        category="Join",
        symptom="Integrated Silver or executive dashboard output fails because of duplicate or ambiguous columns.",
        likely_cause="Multiple joined tables contain the same column names, such as region, incident_count, ingestion_ts, or source_file.",
        investigation="Inspect df.columns before and after joins and identify repeated names.",
        fix="Select only business-relevant columns before joining and rename ambiguous fields.",
        prevention="Prune columns before joins and use explicit aliases for overlapping business metrics.",
        project_example="The integrated Silver join and executive risk dashboard view were fixed by selecting clean input columns before joining.",
    ),
    Row(
        scenario_id="DBG-005",
        category="Unity Catalog view",
        symptom="Creating a persistent Unity Catalog view fails when it references a temporary view.",
        likely_cause="A persistent view cannot depend on a notebook-session temporary view.",
        investigation="Check whether CREATE VIEW references a temp view created with createOrReplaceTempView().",
        fix="Write the dataframe to a managed Delta table first, then create the persistent view from that table.",
        prevention="Use stable tables as sources for persistent Unity Catalog views.",
        project_example="The executive risk dashboard view was created on top of gold_executive_energy_risk_dashboard_base instead of a temp view.",
    ),
]

debugging_df = spark.createDataFrame(debugging_scenarios)

display(debugging_df)

In [0]:
category_summary_df = (
    debugging_df
    .groupBy("category")
    .count()
    .orderBy("category")
)

display(category_summary_df)

In [0]:
print("Debugging scenarios documented:", debugging_df.count())

print("Key Day 5 debugging message:")
print("The project was not debugged by changing code randomly.")
print("Issues were classified first: path, schema, module cache, join, or Unity Catalog view problem.")
print("Then the investigation and fix were documented so the same issue can be prevented later.")